In [1]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saimo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\saimo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
df = pd.read_csv("IMDB Dataset.csv")  # change name if needed

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
print("Shape:", df.shape)
print(df['sentiment'].value_counts())

df.sample(1000)

Shape: (50000, 3)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment,clean_review
47935,This films makes no pretentious efforts to hid...,positive,film make pretenti effort hide true genr campi...
9322,"The only good thing about ""People I Know"" is t...",negative,good thing peopl know serv perfect exampl movi...
37949,the real plot...<br /><br />A group of post-Ci...,negative,real plotbr br group postcivil war prostitut s...
2081,I first saw this mini-series as a child and th...,positive,first saw miniseri child though child longer s...
28876,i love this movie. it focuses on both issues: ...,positive,love movi focus issu realiti fantasi realiti h...
...,...,...,...
37701,...and my reasons for which are simple- there ...,positive,reason simpl mani great film present discuss d...
26972,(This review will have some very obvious spoil...,negative,review obviou spoiler bewarebr br friend broug...
29895,"I won't describe the story, as that has been d...",negative,wont describ stori done elsewher great clive o...
45030,This is just a short comment but I stumbled on...,positive,short comment stumbl onto movi chanc love act ...


In [8]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # lowercase
    text = text.lower()
    
    # remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # remove numbers
    text = re.sub(r'\d+', '', text)
    
    # tokenization
    words = text.split()
    
    # remove stopwords + stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    
    return " ".join(words)

In [9]:
df['clean_review'] = df['review'].apply(clean_text)

In [10]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [11]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [13]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [14]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

In [15]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

In [16]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

In [17]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\n")

In [18]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")

--- Logistic Regression ---
Accuracy: 0.8853
Precision: 0.8739431206764028
Recall: 0.9025600317523318
F1 Score: 0.8880210875720004


--- Naive Bayes ---
Accuracy: 0.8465
Precision: 0.8433947471579772
Recall: 0.8539392736654098
F1 Score: 0.8486342569766295


--- Decision Tree ---
Accuracy: 0.7162
Precision: 0.7225480283114257
Recall: 0.7090692597737647
F1 Score: 0.7157451923076923




In [19]:
results = {
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
}

pd.DataFrame(results)

,Model,Accuracy,F1 Score
0,Logistic Regression,0.8853,0.888021
1,Naive Bayes,0.8465,0.848634
2,Decision Tree,0.7162,0.715745
